# FedGKT baselines — Colab training

Trains one pyKT baseline on the locked FedGKT split and scores it the same way FedGKT was scored.

**Before starting**
1. `Runtime → Change runtime type → T4 GPU`. pyKT's dataloader will not run without a GPU.
2. Upload `fedgkt_baselines_staging.zip` to the top level of your Google Drive (`MyDrive`).

**Order**
1. Run cells 1–5 once per session (settings, GPU check, Drive, install, unzip).
2. Cell 6 — **smoke test**: 2 epochs, into a separate `<model>_smoke` folder. Confirms everything works.
3. Cell 7 — **full run**: up to 100 epochs with early stopping. If Colab disconnects, reconnect, run cells 1–5 again, then cell 7 again — it resumes from the last finished epoch.
4. Cell 8 — show the results.

Everything is saved to Google Drive under `MyDrive/fedgkt_baselines_runs/`, so nothing is lost if the session ends. Keep this tab open and visible while training — it reduces idle disconnects.

## 1. Settings
Change `MODEL` to train a different baseline. Each model has its own driver in `drivers/` and its own folder on Drive.

In [ ]:
MODEL = 'akt'   # one of: dkt, dkvmn, sakt, akt, gkt   (only dkt exists so far)

ZIP_PATH = '/content/drive/MyDrive/fedgkt_baselines_staging.zip'
RUN_ROOT = '/content/drive/MyDrive/fedgkt_baselines_runs'   # must match the drivers' default
EXTRACT_TO = '/content/staging_extract'

# FedGKT's canonical test result, first interaction dropped (27,871 predictions)
# -- the numbers the baselines are compared against in the head-to-head table.
FEDGKT_TEST_MACRO_N_MINUS_1 = 0.6820
FEDGKT_TEST_POOLED_N_MINUS_1 = 0.7554

## 2. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No GPU. Go to Runtime -> Change runtime type -> T4 GPU, then run this cell again. "
    "(If Colab says no GPU is available, the free quota may be used up for now -- try later.)"
)
print("GPU :", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Install pyKT 0.0.38

Installed with `--no-deps` on purpose. pyKT's code only needs numpy, pandas, scikit-learn and torch, which Colab already has. Letting pip resolve dependencies could replace Colab's GPU build of torch. (pyKT lists `wandb` as a dependency, but nothing in the package ever imports it.)

pyKT 0.0.38 contains a stray `from turtle import forward`, which needs tkinter. This cell installs tkinter only if it's missing.

In [ ]:
!pip install -q pykt-toolkit==0.0.38 --no-deps

try:
    import tkinter  # noqa: F401
except ModuleNotFoundError:
    print("tkinter missing -- installing (needed only for pyKT's stray turtle import)")
    !apt-get -qq install -y python3-tk > /dev/null

import importlib.metadata as md_
v = md_.version('pykt-toolkit')
assert v == '0.0.38', f"Expected pyKT 0.0.38 (the version everything was verified against), got {v}"
from pykt.models import init_model  # noqa: F401  -- fails loudly here if anything is wrong
print("pyKT", v, "installed and importable")

## 5. Unzip the staging folder

Colab's own disk is wiped every session, so this runs each time. Works whether the zip contains the `fedgkt_baselines_staging` folder itself or just its contents.

Also clears pyKT's cached `.pkl` files. pyKT caches each CSV the first time it reads it and ignores the CSV afterwards — clearing makes sure it always reads the CSVs that came in the zip.

In [ ]:
import os, zipfile, shutil, glob

assert os.path.exists(ZIP_PATH), (
    f"Zip not found at {ZIP_PATH}. Upload fedgkt_baselines_staging.zip to the top of your Drive."
)
if os.path.exists(EXTRACT_TO):
    shutil.rmtree(EXTRACT_TO)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_TO)

STAGING = None
for root, dirs, files in os.walk(EXTRACT_TO):
    if {'drivers', 'common', 'pykt_sequences'} <= set(dirs):
        STAGING = root
        break
assert STAGING, "Couldn't find a folder containing drivers/, common/ and pykt_sequences/ inside the zip."

for p in glob.glob(os.path.join(STAGING, 'pykt_sequences', '*.pkl')):
    os.remove(p)
    print("cleared stale cache:", os.path.basename(p))

driver = os.path.join(STAGING, 'drivers', f'run_{MODEL}.py')
assert os.path.exists(driver), f"No driver for '{MODEL}' yet: {driver}"

print("Staging folder:", STAGING)
for sub in ['common', 'drivers', 'pykt_sequences']:
    print(f"  {sub}/:", sorted(os.listdir(os.path.join(STAGING, sub))))

## 6. Smoke test — 2 epochs

Writes to `MyDrive/fedgkt_baselines_runs/<model>_smoke/`, **never** to the real run's folder, so it can't be mistaken for (or resumed into) the real run.

The full console output is also saved to `MyDrive/fedgkt_baselines_runs/logs/`. The low scores after 2 epochs are expected — this only checks that everything runs.

In [ ]:
import datetime
LOG_DIR = f'{RUN_ROOT}/logs'
os.makedirs(LOG_DIR, exist_ok=True)
log = f"{LOG_DIR}/{MODEL}_smoke_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt"

!cd "{STAGING}" && (nvidia-smi --query-gpu=name,memory.total --format=csv; python -u drivers/run_{MODEL}.py --smoke) 2>&1 | tee "{log}"
print("\nlog saved:", log)

## 7. Full run

Up to 100 epochs, early stopping after 5 without improvement in validation macro AUC.

**If Colab disconnects:** reconnect, re-run cells 1–5, then run this cell again. It detects the saved state and prints `RESUMING from epoch N`. Each session gets its own log file, so earlier logs are never overwritten.

In [ ]:
import datetime
LOG_DIR = f'{RUN_ROOT}/logs'
os.makedirs(LOG_DIR, exist_ok=True)
log = f"{LOG_DIR}/{MODEL}_full_{datetime.datetime.now():%Y%m%d_%H%M%S}.txt"

!cd "{STAGING}" && (nvidia-smi --query-gpu=name,memory.total --format=csv; python -u drivers/run_{MODEL}.py) 2>&1 | tee "{log}"
print("\nlog saved:", log)

## 8. Results

In [ ]:
import json
path = f'{RUN_ROOT}/{MODEL}/results.json'
assert os.path.exists(path), f"No results yet at {path} -- has the full run (cell 7) finished?"
r = json.load(open(path))

print(f"{MODEL.upper()}")
print(f"  stopped        : {r['stopped_reason']}  after {r['epochs_run']} epochs")
print(f"  best epoch     : {r['best_epoch']}   (val macro AUC {r['best_val_macro_auc']:.4f})")
print(f"  test predictions scored: {r['test_n_scored']:,}   (expected 27,871)")
print()
print(f"                  {MODEL.upper():>8}   FedGKT")
print(f"  test macro AUC  {r['test_macro_auc']:8.4f}   {FEDGKT_TEST_MACRO_N_MINUS_1:.4f}")
print(f"  test pooled AUC {r['test_pooled_auc']:8.4f}   {FEDGKT_TEST_POOLED_N_MINUS_1:.4f}")
print()
print("(Both columns scored on the same 27,871 predictions: each student's first interaction")
print(" excluded, best epoch chosen by validation macro AUC, no shuffling.)")